## Concept focus — Measurement-first performance work

Performance work goes wrong when people optimize from instinct instead of evidence. This exercise trains the habit of predicting cost, measuring with tools, and then checking whether your intuition was actually correct.

```text
hunch -> measure baseline -> identify hotspot -> change code -> measure again

No benchmark = no performance claim
```

### How to think about it
Performance is an engineering investigation, not a guessing contest. Always separate anecdote from evidence, and remember that the slowest-looking line is not always the real bottleneck once you profile the whole path.

### Visual references and further study
- [timeit documentation](https://docs.python.org/3/library/timeit.html)
- [cProfile documentation](https://docs.python.org/3/library/profile.html)
- [pyperf documentation](https://pyperf.readthedocs.io/en/latest/)
- [Python performance talk search](https://www.youtube.com/results?search_query=python+profiling+performance)

---

# Module 23 — Performance and Profiling

## Exercise 23.1 — Rank ten operations, then measure

Write your predicted ranking (1 = fastest) BEFORE running. Then score yourself.
Getting the ORDER right matters more than the numbers -- the order is what you
use when reading code, and the numbers change with hardware and version.
Run:  python ex01_predict.py

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.

---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 1. The order of operations

Work down this list. Each step is 10 to 100 times more valuable than the one
below it.

| # | Question | Typical gain |
|---|---|---|
| 1 | Do you need to do this at all? | ∞ |
| 2 | Is the algorithm right? (Module 05) | 10–10,000x |
| 3 | Is it doing I/O it could avoid, batch, or cache? | 10–1000x |
| 4 | Is the data structure right? (Module 05) | 10–100x |
| 5 | Can the work be done by a C library (NumPy, pandas)? | 10–100x |
| 6 | Can it be parallel? (Module 21) | up to core count |
| 7 | Micro-optimise Python | 1.1–2x |
| 8 | Rewrite in C / Rust / Cython | 10–100x, at great cost |

**Almost everybody starts at 7.** The N+1 query at step 3 costs a thousand times
more than every local variable lookup in the file, and fixing it is a two-line
change.

---

## Concept 2. Measuring correctly

In [ ]:
import timeit
timeit.repeat("f(x)", setup="from __main__ import f, x", number=1000, repeat=5)

Four rules, each of which is the difference between a benchmark you can believe
and one you cannot:

**Take the minimum, not the mean.** Noise is one-sided: nothing makes code
faster than it can run, but a GC pause, another process, CPU frequency scaling
or a cache miss can all make it slower. The minimum estimates the true cost; the
mean measures how busy your laptop was.

**Do not let setup leak into the measurement.** `timeit`'s `setup` is not timed —
put everything you are not measuring there.

**Beware constant folding.** `timeit("1 + 2")` measures nothing; the compiler
folded it (Module 01).

**Measure the thing you actually do, at the rate you actually do it.** A 50x
ratio on a 7-microsecond operation is not a performance problem (Module 02).

For anything wall-clock, `time.perf_counter()`. Never `time.time()` (Module 19).

---

## Concept 3. `cProfile`: where the time goes

```bash
python -m cProfile -o out.prof app.py
python -m pstats out.prof
```

In [ ]:
import cProfile, pstats
with cProfile.Profile() as prof:
    main()
pstats.Stats(prof).sort_stats("cumulative").print_stats(20)

**Two columns, two meanings, and confusing them wastes hours:**

- **`tottime`** — time in this function, *excluding* callees. Points at the leaf
  actually burning CPU.
- **`cumtime`** — time in this function *and everything it called*. Points at
  the branch of the program responsible.

Sort by `cumulative` to find *what area* is slow; sort by `tottime` to find
*which function* to change.

**`cProfile` adds per-call overhead**, so it distorts programs dominated by many
tiny calls. For a production process, use a **sampling** profiler:

```bash
py-spy top --pid 1234              # live, no restart, negligible overhead
py-spy record -o flame.svg --pid 1234
py-spy dump --pid 1234             # stack traces of every thread -- for hangs
```

`py-spy dump` on a hung process is the single most useful debugging tool in this
module.

**Flame graphs** read like this: width is time, stacking is call depth. A wide
plateau is where the time is. A tall narrow spike is deep recursion and usually
not your problem.

---

## Concept 5. Where Python's time actually goes

Rough costs on a modern CPU. **The ratios are the lesson; the absolute numbers
vary by hardware and by version** — 3.11's specialising interpreter changed
several of these substantially, and "zero-cost" exception handling made an
untaken `try` genuinely free.

| Operation | Approx. |
|---|---|
| Local variable read | ~10 ns |
| Global variable read | ~15 ns |
| Attribute access | ~20 ns |
| Function call | **~60 ns** |
| Method call | ~70 ns |
| Creating an object | ~80 ns |
| Dict lookup | ~25 ns |
| List append (amortized) | ~30 ns |
| `try` with no exception | ~0 ns (free since 3.11) |
| Raising and catching | ~50-200 ns |
| A NumPy op per element | ~1 ns |

**A Python function call is the expensive primitive.** That is why a
comprehension beats `map(lambda ...)` (Module 01), why hot loops sometimes
inline, and why NumPy wins — one call processing a million elements instead of a
million calls.

The techniques that actually help, in a loop a profiler has flagged:

In [ ]:
local_len = len                       # bind globals/builtins to locals
result = [x * 2 for x in data]        # comprehension over an explicit loop
"".join(parts)                        # never += in a loop (Module 03)
seen = set(other)                     # membership (Module 05)
if x in seen: ...

And the one that matters most: **do less work**. Hoist invariants out of loops,
avoid recomputing, and cache what is pure (Module 15).

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: The order of operations
- Section 2: Measuring correctly
- Section 3: `cProfile`: where the time goes
- Section 4: Memory
- Section 5: Where Python's time actually goes
- Section 6: Vectorising
- Section 7: Caching
- Section 8: When to stop, and when to reach for C

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

import timeit

PREDICTIONS = """
Rank these 1 (fastest) to 10 (slowest), per operation:

  __ read a local variable
  __ read a global variable
  __ read an attribute (obj.x)
  __ call a function that does nothing
  __ call a method that does nothing
  __ create a small object (a 3-field class instance)
  __ dict lookup by string key
  __ list append
  __ enter a try block where nothing raises
  __ raise and catch an exception

Then predict the RATIO between the fastest and the slowest. Most people
underestimate it by an order of magnitude.
"""

SETUP = """
class Thing:
    __slots__ = ('x', 'y', 'z')
    def __init__(self): self.x = self.y = self.z = 1
    def method(self): pass

def function(): pass

obj = Thing()
d = {'key': 1}
lst = []
g = 1
"""

CASES = [
    ("local variable read",   "x = 1\nx"),
    ("global variable read",  "g"),
    ("attribute read",        "obj.x"),
    ("function call",         "function()"),
    ("method call",           "obj.method()"),
    ("object creation",       "Thing()"),
    ("dict lookup",           "d['key']"),
    ("list append",           "lst.append(1)"),
    ("try, no exception",     "try:\n    pass\nexcept ValueError:\n    pass"),
    ("raise and catch",       "try:\n    raise ValueError()\nexcept ValueError:\n    pass"),
]

---

## `measure`

_measure_

In [ ]:
def measure() -> None:
    results: list[tuple[str, float]] = []
    for label, stmt in CASES:
        best = min(timeit.repeat(stmt, SETUP, number=200_000, repeat=5))
        results.append((label, best / 200_000 * 1e9))

    results.sort(key=lambda kv: kv[1])
    fastest = results[0][1]
    print(f"\n  {'rank':<6}{'operation':<24}{'ns/op':>10}{'relative':>12}")
    print("  " + "-" * 52)
    for rank, (label, ns) in enumerate(results, start=1):
        print(f"  {rank:<6}{label:<24}{ns:>10.1f}{ns / fastest:>11.1f}x")

    print(
        "\n  THE ONE THAT MATTERS: compare the function-call row with the\n"
        "  local-variable row. That ratio is why a comprehension beats\n"
        "  map(lambda ...), why hot loops sometimes bind builtins to locals,\n"
        "  and above all why NumPy wins -- one call over a million elements\n"
        "  instead of a million calls."
    )

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
# TODO 1: score your ranking. How many of the ten did you place correctly?

# TODO 2: the try/except row surprises people. Explain the result -- what does
#         entering a try block cost when nothing raises, and why? (Module 24
#         and the `dis` module will tell you.)

# TODO 3: add three more rows and predict each first:
#         - a list comprehension over 100 items
#         - the same with map(lambda ...)
#         - the same with a plain for loop and append
#         Explain the ordering in terms of the function-call cost.

# TODO 4: now the question that matters. Take the SLOWEST operation here and
#         compute how many times per second you would have to do it before it
#         accounted for one percent of a 200ms request. Then say what that
#         means for how much time you should spend micro-optimising.


if __name__ == "__main__":
    print(PREDICTIONS)
    measure()

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.